In [38]:
!apt-get update -qq
!apt-get install -qq libspatialindex-dev

!pip install osmnx folium networkx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [42]:
from google.colab import drive
drive.mount('/content/drive')

df_url = '/content/drive/MyDrive/관광지.csv'

region_url = '/content/drive/MyDrive/국민여행조사_지역코드2.csv'

infra_url = '/content/drive/MyDrive/교통인프라지수_최종(3).csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

df = pd.read_csv(df_url)

region_code = pd.read_csv(region_url)

infra_df = pd.read_csv(infra_url)

In [4]:
df_filtered = df.copy()
df = df_filtered[~df_filtered['CL_NM'].isin(['정보화마을','항공사/여행사', '영어마을'])].copy()


In [5]:
region_code['시도명']  = region_code['시도명'].str.strip()

In [6]:
region_code = region_code.rename(columns={'지역명.1':'region_name'})


In [7]:
df_filtered = df[['POI_ID','POI_NM','MLSFC_NM', 'CL_NM', 'CTPRVN_NM', 'SIGNGU_NM', 'CL_CD','LC_LO', 'LC_LA','GID_CD']]

In [8]:
df_filtered['SIGNGU_NM'] = df_filtered['SIGNGU_NM'].str.split().str[0]

/tmp/ipython-input-3618527517.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['SIGNGU_NM'] = df_filtered['SIGNGU_NM'].str.split().str[0]


In [9]:
df_filtered=df_filtered.fillna('세종시')

In [10]:
df_filtered_copy = df_filtered.copy()

region_code['지역 코드'] = region_code['지역 코드'].astype(str).str.zfill(5)

df_filtered_copy = df_filtered_copy.merge(
    region_code[['시도명', '지역명', '지역 코드']],
    left_on=['CTPRVN_NM', 'SIGNGU_NM'],
    right_on=['시도명', '지역명'],
    how='left'
)

df_filtered_copy['시도 코드']   = df_filtered_copy['지역 코드'].str[:2]
df_filtered_copy['시군구 코드'] = df_filtered_copy['지역 코드'].str[2:]

In [11]:
df_filtered_copy['지역 코드']    = df_filtered_copy['지역 코드'].astype(str)
infra_df           ['시군구코드'] = infra_df           ['시군구코드'].astype(str)


df_filtered_copy2 = df_filtered_copy.merge(
    infra_df[['시군구코드','교통인프라지수_norm']],
    left_on  = '지역 코드',
    right_on = '시군구코드',
    how       = 'left'
)

df_filtered_copy2 = df_filtered_copy2.drop(columns=['시군구코드'])


In [12]:
df_filtered_copy2 = df_filtered_copy2.drop('시도명', axis=1)
df_filtered_copy2 = df_filtered_copy2.drop('지역명', axis=1)

In [13]:
missing_rows = df_filtered_copy2[df_filtered_copy2['지역 코드'].isna()]
print(len(missing_rows['SIGNGU_NM'].unique()))

0


In [14]:
df_filtered_copy2['지역 코드'] = pd.to_numeric(df_filtered_copy2['지역 코드'], errors='coerce')


prov_map = {
    '대구광역시': 22,
    '인천광역시': 23
}

df_filtered_copy2['지역 코드'] = df_filtered_copy2['지역 코드'].fillna(
    df_filtered_copy2['CTPRVN_NM'].map(prov_map)
)

df_filtered_copy2['지역 코드'] = df_filtered_copy2['지역 코드'].astype(int).astype(str).str.zfill(2)


df_filtered_copy2['시도 코드']   = df_filtered_copy2['지역 코드'].str[:2]
df_filtered_copy2['시군구 코드'] = df_filtered_copy2['지역 코드'].str[2:]

In [15]:
df_filtered_copy2['교통인프라지수_norm'] = df_filtered_copy2['교통인프라지수_norm'].fillna('0')

In [16]:
df_filtered_copy2['교통인프라지수_norm'] = df_filtered_copy2['교통인프라지수_norm'].astype(float)

# 지역 선정

## 교통 인프라 부족 지역 선정

In [17]:
infra_df.tail(10)

,Unnamed: 0,시군구코드,교통인프라지수_norm
217,181,36460,-0.564304
218,156,35330,-0.571005
219,212,38090,-0.572747
220,159,35360,-0.572994
221,157,35340,-0.578193
222,219,38350,-0.583268
223,183,36480,-0.586901
224,127,33340,-0.616062
225,197,37340,-0.617247
226,182,36470,-0.618003


In [18]:
total_sites = df_filtered_copy2.groupby("지역 코드").size()
total_sites

,0
지역 코드,
11010,343
11020,176
11030,63
11040,46
11050,42
...,...
38380,148
38390,126
38400,144


In [19]:
cat_counts = (df_filtered_copy2
              .groupby(["지역 코드","CL_NM"])
              .size()
              .unstack(fill_value=0))

cat_counts.columns

Index(['N', '고택/생가/민속마을', '관광농원/허브마을', '관광안내소/매표소', '국보', '궁궐/종묘',
       '글램핑코리아(캠핑)', '동물원', '드라마/영화촬영지', '먹거리/패션거리', '보물', '비/탑/문/각',
       '서원/향교/서당', '성/성터', '식물원', '아쿠아리움/대형수족관', '야영장', '온천지역', '왕릉/고분',
       '유명관광지', '유명사적/유적지', '일반관광지', '일반유원지/일반놀이공원', '잼핑홀리데이(캠핑)', '지역축제',
       '천연기념물', '캠핑장', '캠핑홀리데이(캠핑)', '테마공원/대형놀이공원', '팜스테이', '폭포/계곡', '해수욕장',
       '휴양림/수목원'],
      dtype='object', name='CL_NM')

In [20]:
import geopandas as gpd
import pandas as pd
import glob

def load_shp(fp):
    gdf = gpd.read_file(fp, encoding="euc-kr")
    return gdf.rename(columns={
        "ADM_SECT_C": "shp_code",
        "SGG_NM":     "region_name"
    })

shps = glob.glob("/content/drive/MyDrive/geo/SGG_*.shp")
gdfs = [load_shp(p) for p in shps]

regions = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
).to_crs(epsg=4326)

region_code = region_code.rename(columns={"지역 코드":"region_code"})
region_code["region_name"] = region_code["region_name"].str.strip()

regions['region_name'] = regions['region_name'].str.strip()
region_code  ['region_name'] = region_code  ['region_name'].str.strip()



/usr/local/lib/python3.11/dist-packages/geopandas/array.py:1755: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as KGD2002_Central_Belt_2010 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


In [21]:
kb = region_code[region_code['region_name'].str.startswith('전북특별차지도')]
print(kb[['region_name','region_code']].drop_duplicates().sort_values('region_code'))

Empty DataFrame
Columns: [region_name, region_code]
Index: []


In [22]:
regions['region_name'] = (
    regions['region_name']
    .str.split()
    .str[:2]
    .str.join(' ')
)

In [23]:
regions['region_name'] = regions['region_name'] \
    .str.replace(r'^전북특별차지도', '전북특별자치도', regex=True)

In [24]:
regions.loc[
    regions['region_name']=='인천광역시 미추홀구',
    'region_name'
] = '인천광역시 남구'

In [25]:
region_code.loc[
    region_code['region_name']=='경상북도 군위군',
    'region_name'
] = '대구광역시 군위군'

In [26]:
set(regions['region_name']) - set(region_code['region_name'])


set()

In [27]:
regions = regions.merge(
    region_code[["region_name","region_code"]],
    on="region_name",
    how="left"
)

## 교통 인프라 보완 지역 선정

In [28]:
import geopandas as gpd
from shapely.geometry import Point

sites = df_filtered_copy2.copy()

sites['geometry'] = sites.apply(
    lambda r: Point(r['LC_LO'], r['LC_LA']),
    axis=1
)

sites_gdf = gpd.GeoDataFrame(
    sites,
    geometry='geometry',
    crs="EPSG:4326"
)

In [29]:
regions['centroid']      = regions.geometry.centroid
regions['centroid_lat']  = regions.centroid.y
regions['centroid_lon']  = regions.centroid.x

/tmp/ipython-input-3768008211.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  regions['centroid']      = regions.geometry.centroid
/tmp/ipython-input-3768008211.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  regions['centroid_lat']  = regions.centroid.y
/tmp/ipython-input-3768008211.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  regions['centroid_lon']  = regions.centroid.x


In [30]:
infra_subset = infra_df.rename(columns={'시군구코드':'region_code'})[
    ['region_code','교통인프라지수_norm']
]

In [31]:
regions = regions.merge(
    infra_subset,
    on='region_code',
    how='left'
)

In [32]:
target_codes = ['36470','38340','37370','33370']
targets = regions[regions['region_code'].isin(target_codes)].copy()

In [33]:
regions.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   shp_code      252 non-null    object  
 1   region_name   252 non-null    object  
 2   SGG_OID       237 non-null    float64 
 3   COL_ADM_SE    252 non-null    object  
 4   geometry      252 non-null    geometry
 5   region_code   252 non-null    object  
 6   centroid      252 non-null    geometry
 7   centroid_lat  252 non-null    float64 
 8   centroid_lon  252 non-null    float64 
 9   교통인프라지수_norm  250 non-null    float64 
dtypes: float64(4), geometry(2), object(4)
memory usage: 19.8+ KB


# 관광 루트 선정

In [34]:
print(regions['region_code'].dtype)
regions['region_code'] = pd.to_numeric(regions['region_code'], errors='coerce').astype('Int64')

object


In [35]:
!pip install polyline

## 노선만 표시된 지도

In [43]:
route_1 = pd.read_csv('/content/drive/MyDrive/onroute_22020_37370_노선.csv')
route_2 = pd.read_csv('/content/drive/MyDrive/onroute_33040_33370_노선.csv')
route_3 = pd.read_csv('/content/drive/MyDrive/onroute_38110_38340_노선.csv')
route_4 = pd.read_csv('/content/drive/MyDrive/onroute_36010_36470_노선.csv')


In [44]:
route_1

,POI_ID,POI_NM,src_code,dst_code,src_region,dst_region,lon,lat,dist_on_route_m
0,493475,대구역관광안내소,22020,37370,대구광역시 동구,경상북도 고령군,128.595726,35.875751,15221.60151
1,1121626,동성로로데오거리,22020,37370,대구광역시 동구,경상북도 고령군,128.598019,35.867261,15237.05618
2,5251480,동성로,22020,37370,대구광역시 동구,경상북도 고령군,128.593758,35.867154,16342.48548
3,992388,약전골목,22020,37370,대구광역시 동구,경상북도 고령군,128.589943,35.867820,16634.09833
4,3317247,두류젊음의거리,22020,37370,대구광역시 동구,경상북도 고령군,128.553499,35.855928,24826.70853
5,3809700,이월드인생꽃사진관,22020,37370,대구광역시 동구,경상북도 고령군,128.565439,35.854875,24845.92139
6,5428615,장기동먹거리촌,22020,37370,대구광역시 동구,경상북도 고령군,128.530267,35.841906,26564.10209
7,1200867,서부정류장막창골목,22020,37370,대구광역시 동구,경상북도 고령군,128.558698,35.837629,26805.80469
8,620841,영남조경,22020,37370,대구광역시 동구,경상북도 고령군,128.480568,35.794295,32959.47998
9,3552635,용연사벚꽃길,22020,37370,대구광역시 동구,경상북도 고령군,128.473305,35.786598,34130.13700


In [51]:
import osmnx as ox
import networkx as nx
import folium
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import linemerge
import math
import gc
import pandas as pd

PAIR_TO_DF = {
    (22020, 37370): route_1,
    (33040, 33370): route_2,
    (38110, 38340): route_3,
    (36010, 36470): route_4,
}
interest_pairs = list(PAIR_TO_DF.keys())

def haversine(lon1, lat1, lon2, lat2):
    R=6371000
    φ1,φ2 = math.radians(lat1), math.radians(lat2)
    Δφ,Δλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

def load_graph_segment(src, dst, pad_deg=0.2):
    north = max(src.centroid_lat, dst.centroid_lat) + pad_deg
    south = min(src.centroid_lat, dst.centroid_lat) - pad_deg
    east  = max(src.centroid_lon, dst.centroid_lon) + pad_deg
    west  = min(src.centroid_lon, dst.centroid_lon) - pad_deg

    try:
        G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
        if G.edges:
            return G
    except ValueError:
        pass

    mid_lat = (src.centroid_lat + dst.centroid_lat)/2
    mid_lon = (src.centroid_lon + dst.centroid_lon)/2
    half = haversine(src.centroid_lon, src.centroid_lat,
                     dst.centroid_lon, dst.centroid_lat)/2
    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+10000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+50000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    return nx.MultiDiGraph()

def route_linestring(G, route):
    try:
        geom = ox.utils_graph.route_to_geometry(G, route)
        if geom.geom_type == "LineString":
            return geom
        if geom.geom_type == "MultiLineString":
            return linemerge(geom)
    except Exception:
        pass
    coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in route]
    if len(coords) >= 2:
        return LineString(coords)

    x, y = coords[0]
    eps = 1e-6
    return LineString([(x, y), (x+eps, y+eps)])

def _add_polyline(m, geom, color="black", weight=4, opacity=0.9):
    if geom.geom_type == "LineString":
        folium.PolyLine([[y, x] for x, y in geom.coords],
                        color=color, weight=weight, opacity=opacity).add_to(m)
    elif geom.geom_type == "MultiLineString":
        for g in geom.geoms:
            if g.geom_type == "LineString":
                folium.PolyLine([[y, x] for x, y in g.coords],
                                color=color, weight=weight, opacity=opacity).add_to(m)

def build_route(G, src, dst, via_pois=None, weight="travel_time"):
    orig = ox.distance.nearest_nodes(G, src.centroid_lon, src.centroid_lat)
    dest = ox.distance.nearest_nodes(G, dst.centroid_lon, dst.centroid_lat)

    if via_pois is not None and not via_pois.empty:
        via_nodes = []
        used_rows = []
        for idx, p in via_pois.iterrows():
            try:
                n = ox.distance.nearest_nodes(G, p.geometry.x, p.geometry.y)
                via_nodes.append(n)
                used_rows.append(idx)
            except Exception:
                pass
        via_pois = via_pois.loc[used_rows]
        seq = [orig] + via_nodes + [dest]
        full=[]
        for u,v in zip(seq[:-1], seq[1:]):
            seg = nx.shortest_path(G, u, v, weight=weight)
            full.extend(seg[:-1])
        full.append(seq[-1])
        return full, via_pois

    return nx.shortest_path(G, orig, dest, weight=weight), via_pois

def draw_route_map(G, route, src, dst, sites_gdf=None,
                   via_pois=None, buffer_km=2, iso_time=1800):

    line = route_linestring(G, route)

    sub = nx.ego_graph(G, route[0], radius=iso_time, distance="travel_time")
    iso_pts = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in sub.nodes]
    if len(iso_pts) > 2:
        iso = Polygon(iso_pts).convex_hull
    elif len(iso_pts) == 2:
        iso = LineString(iso_pts).buffer(buffer_km/111)
    else:
        iso = Point(iso_pts[0]).buffer(buffer_km/111)

    m = folium.Map(
        location=[(src.centroid_lat+dst.centroid_lat)/2,
                  (src.centroid_lon+dst.centroid_lon)/2],
        zoom_start=9
    )

    folium.GeoJson(iso, style_function=lambda _: {
        "fillColor":"#3186cc","color":"#3186cc","fillOpacity":0.18
    }).add_to(m)

    _add_polyline(m, line, color="black", weight=4, opacity=0.9)

    for city,col in [(src,"red"),(dst,"blue")]:
        folium.CircleMarker(
            [city.centroid_lat, city.centroid_lon],
            radius=6, color=col, fill=True,
            tooltip=city.region_name
        ).add_to(m)

    if via_pois is not None and not via_pois.empty:
        for _,p in via_pois.iterrows():
            folium.Marker(
                [p.geometry.y,p.geometry.x],
                icon=folium.Icon(color="orange", icon="star"),
                tooltip=f"경유: {p.POI_NM}"
            ).add_to(m)

    return m, line, iso

for src_code, dst_code in interest_pairs:
    src = regions.loc[regions.region_code==src_code].iloc[0]
    dst = regions.loc[regions.region_code==dst_code].iloc[0]

    pad = 0.2
    Gp = load_graph_segment(src, dst, pad_deg=pad)
    if not Gp.edges:
        print(f"Skip {src.region_name}→{dst.region_name}: no edges")
        continue

    Gp = ox.add_edge_speeds(Gp, fallback=30)
    Gp = ox.add_edge_travel_times(Gp)

    via_df = PAIR_TO_DF[(src_code, dst_code)].copy()

    for col in ("lon","lat","POI_NM"):
        if col not in via_df.columns:
            raise ValueError(f"{(src_code, dst_code)}: '{col}' 컬럼이 route CSV에 필요합니다.")
    via_df["lon"] = pd.to_numeric(via_df["lon"], errors="coerce")
    via_df["lat"] = pd.to_numeric(via_df["lat"], errors="coerce")
    via_df = via_df.dropna(subset=["lon","lat"])
    via_df["geometry"] = via_df.apply(lambda r: Point(r["lon"], r["lat"]), axis=1)

    route, used_via = build_route(Gp, src, dst, via_pois=via_df, weight="travel_time")

    buffer_km, iso_time = (5, 3600) if not used_via.empty else (2, 1800)
    m, line, iso = draw_route_map(Gp, route, src, dst, sites_gdf=None,
                                  via_pois=used_via,
                                  buffer_km=buffer_km,
                                  iso_time=iso_time)
    display(m)

    del Gp, route, m, line, iso, via_df, used_via
    gc.collect()

/usr/local/lib/python3.11/dist-packages/osmnx/routing.py:577: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  edges[["highway", "speed_kph"]].set_index("highway").iloc[:, 0].fillna(hwy_speed_avg)


ValueError: (36010, 36470): 'lon' 컬럼이 route CSV에 필요합니다.

# 대구 -- 고령

In [56]:
import osmnx as ox
import networkx as nx
import folium
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import linemerge
import math
import gc
import pandas as pd

PAIR_TO_DF = {
    (22020, 37370): route_1
}
interest_pairs = list(PAIR_TO_DF.keys())

def haversine(lon1, lat1, lon2, lat2):
    R=6371000
    φ1,φ2 = math.radians(lat1), math.radians(lat2)
    Δφ,Δλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

def load_graph_segment(src, dst, pad_deg=0.2):
    north = max(src.centroid_lat, dst.centroid_lat) + pad_deg
    south = min(src.centroid_lat, dst.centroid_lat) - pad_deg
    east  = max(src.centroid_lon, dst.centroid_lon) + pad_deg
    west  = min(src.centroid_lon, dst.centroid_lon) - pad_deg

    try:
        G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
        if G.edges:
            return G
    except ValueError:
        pass

    mid_lat = (src.centroid_lat + dst.centroid_lat)/2
    mid_lon = (src.centroid_lon + dst.centroid_lon)/2
    half = haversine(src.centroid_lon, src.centroid_lat,
                     dst.centroid_lon, dst.centroid_lat)/2
    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+10000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+50000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    return nx.MultiDiGraph()

def route_linestring(G, route):
    try:
        geom = ox.utils_graph.route_to_geometry(G, route)
        if geom.geom_type == "LineString":
            return geom
        if geom.geom_type == "MultiLineString":
            return linemerge(geom)
    except Exception:
        pass

    coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in route]
    if len(coords) >= 2:
        return LineString(coords)

    x, y = coords[0]
    eps = 1e-6
    return LineString([(x, y), (x+eps, y+eps)])

def _add_polyline(m, geom, color="black", weight=5, opacity=0.95):
    if geom.geom_type == "LineString":
        folium.PolyLine([[y, x] for x, y in geom.coords],
                        color=color, weight=weight, opacity=opacity).add_to(m)
    elif geom.geom_type == "MultiLineString":
        for g in geom.geoms:
            if g.geom_type == "LineString":
                folium.PolyLine([[y, x] for x, y in g.coords],
                                color=color, weight=weight, opacity=opacity).add_to(m)

def build_route(G, src, dst, via_pois=None, weight="travel_time"):
    orig = ox.distance.nearest_nodes(G, src.centroid_lon, src.centroid_lat)
    dest = ox.distance.nearest_nodes(G, dst.centroid_lon, dst.centroid_lat)

    if via_pois is not None and not via_pois.empty:
        via_nodes = []
        used_rows = []
        for idx, p in via_pois.iterrows():
            try:
                n = ox.distance.nearest_nodes(G, p.geometry.x, p.geometry.y)
                via_nodes.append(n)
                used_rows.append(idx)
            except Exception:
                pass
        via_pois = via_pois.loc[used_rows]
        seq = [orig] + via_nodes + [dest]
        full=[]
        for u,v in zip(seq[:-1], seq[1:]):
            seg = nx.shortest_path(G, u, v, weight=weight)
            full.extend(seg[:-1])
        full.append(seq[-1])
        return full, via_pois

    return nx.shortest_path(G, orig, dest, weight=weight), via_pois

def draw_route_map(G, route, src, dst, sites_gdf=None,
                   via_pois=None, buffer_km=2, iso_time=1800):

    line = route_linestring(G, route)

    sub = nx.ego_graph(G, route[0], radius=iso_time, distance="travel_time")
    iso_pts = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in sub.nodes]
    if len(iso_pts) > 2:
        iso = Polygon(iso_pts).convex_hull
    elif len(iso_pts) == 2:
        iso = LineString(iso_pts).buffer(buffer_km/111)
    else:
        iso = Point(iso_pts[0]).buffer(buffer_km/111)

    m = folium.Map(
        location=[(src.centroid_lat+dst.centroid_lat)/2,
                  (src.centroid_lon+dst.centroid_lon)/2],
        zoom_start=9
    )

    folium.GeoJson(iso, style_function=lambda _: {
        "fillColor":"#3186cc","color":"#3186cc","fillOpacity":0.18
    }).add_to(m)

    _add_polyline(m, line, color="black", weight=5, opacity=0.95)

    for city,col in [(src,"red"),(dst,"blue")]:
        folium.CircleMarker(
            [city.centroid_lat, city.centroid_lon],
            radius=6, color=col, fill=True,
            tooltip=city.region_name
        ).add_to(m)

    if via_pois is not None and not via_pois.empty:
        for _,p in via_pois.iterrows():
            folium.Marker(
                [p.geometry.y,p.geometry.x],
                icon=folium.Icon(color="orange", icon="star"),
                tooltip=f"경유: {p.POI_NM}"
            ).add_to(m)

    return m, line, iso

for src_code, dst_code in interest_pairs:
    src = regions.loc[regions.region_code==src_code].iloc[0]
    dst = regions.loc[regions.region_code==dst_code].iloc[0]

    PAD_BY_PAIR = {
        (22020, 37370): 0.5,  # 대구 ↔ 고령
        (33040, 33370): 0.5,  # 문제난 구간이면 여기서 조정
    }
    pad = PAD_BY_PAIR.get((src_code, dst_code), 0.2)

    Gp = load_graph_segment(src, dst, pad_deg=pad)
    if not Gp.edges:
        print(f"Skip {src.region_name}→{dst.region_name}: no edges")
        continue

    Gp = ox.add_edge_speeds(Gp, fallback=30)
    Gp = ox.add_edge_travel_times(Gp)

    via_df = PAIR_TO_DF[(src_code, dst_code)].copy()
    for col in ("lon","lat","POI_NM"):
        if col not in via_df.columns:
            raise ValueError(f"{(src_code, dst_code)}: '{col}' 컬럼이 route CSV에 필요합니다.")
    via_df["lon"] = pd.to_numeric(via_df["lon"], errors="coerce")
    via_df["lat"] = pd.to_numeric(via_df["lat"], errors="coerce")
    via_df = via_df.dropna(subset=["lon","lat"])
    via_df["geometry"] = via_df.apply(lambda r: Point(r["lon"], r["lat"]), axis=1)

    route, used_via = build_route(Gp, src, dst, via_pois=via_df, weight="travel_time")

    expect_names = list(via_df["POI_NM"].astype(str))
    used_names = set(used_via["POI_NM"].astype(str)) if used_via is not None and not used_via.empty else set()
    dropped = [nm for nm in expect_names if nm not in used_names]
    if dropped:
        print(f"[경고] 그래프 매칭 실패로 제외된 경유지: {dropped}")

    line_dbg = route_linestring(Gp, route)
    if line_dbg.length < 1e-5:
        print(f"[주의] {src.region_name}→{dst.region_name} 경로가 매우 짧음. pad 확장 재시도")
        pad2 = max(0.5, pad + 0.3)
        Gp2 = load_graph_segment(src, dst, pad_deg=pad2)
        if Gp2.edges:
            Gp2 = ox.add_edge_speeds(Gp2, fallback=30)
            Gp2 = ox.add_edge_travel_times(Gp2)
            route, used_via = build_route(Gp2, src, dst, via_pois=via_df, weight="travel_time")
            Gp = Gp2
            used_names = set(used_via["POI_NM"].astype(str)) if not used_via.empty else set()
            dropped = [nm for nm in expect_names if nm not in used_names]
            if dropped:
                print(f"[경고-재시도 후] 제외된 경유지: {dropped}")

    buffer_km, iso_time = (5, 3600) if used_via is not None and not used_via.empty else (2, 1800)
    m, line, iso = draw_route_map(
        Gp, route, src, dst, sites_gdf=None,
        via_pois=used_via, buffer_km=buffer_km, iso_time=iso_time
    )
    display(m)

    del Gp, route, m, line, iso, via_df, used_via
    gc.collect()

# 청주 -- 음성

In [48]:
import osmnx as ox
import networkx as nx
import folium
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import linemerge
import math
import gc
import pandas as pd

PAIR_TO_DF = {
    (33040, 33370): route_2,
    (38110, 38340): route_3
}
interest_pairs = list(PAIR_TO_DF.keys())

def haversine(lon1, lat1, lon2, lat2):
    R=6371000
    φ1,φ2 = math.radians(lat1), math.radians(lat2)
    Δφ,Δλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

def load_graph_segment(src, dst, pad_deg=0.2):
    north = max(src.centroid_lat, dst.centroid_lat) + pad_deg
    south = min(src.centroid_lat, dst.centroid_lat) - pad_deg
    east  = max(src.centroid_lon, dst.centroid_lon) + pad_deg
    west  = min(src.centroid_lon, dst.centroid_lon) - pad_deg

    try:
        G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
        if G.edges:
            return G
    except ValueError:
        pass

    mid_lat = (src.centroid_lat + dst.centroid_lat)/2
    mid_lon = (src.centroid_lon + dst.centroid_lon)/2
    half = haversine(src.centroid_lon, src.centroid_lat,
                     dst.centroid_lon, dst.centroid_lat)/2
    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+10000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    try:
        G = ox.graph_from_point((mid_lat, mid_lon),
                                dist=half+50000,
                                network_type="drive",
                                simplify=True)
        if G.edges:
            return G
    except ValueError:
        pass

    return nx.MultiDiGraph()

def route_linestring(G, route):
    try:
        geom = ox.utils_graph.route_to_geometry(G, route)
        if geom.geom_type == "LineString":
            return geom
        if geom.geom_type == "MultiLineString":
            return linemerge(geom)
    except Exception:
        pass
    coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in route]
    if len(coords) >= 2:
        return LineString(coords)

    x, y = coords[0]
    eps = 1e-6
    return LineString([(x, y), (x+eps, y+eps)])

def _add_polyline(m, geom, color="black", weight=4, opacity=0.9):
    if geom.geom_type == "LineString":
        folium.PolyLine([[y, x] for x, y in geom.coords],
                        color=color, weight=weight, opacity=opacity).add_to(m)
    elif geom.geom_type == "MultiLineString":
        for g in geom.geoms:
            if g.geom_type == "LineString":
                folium.PolyLine([[y, x] for x, y in g.coords],
                                color=color, weight=weight, opacity=opacity).add_to(m)

def build_route(G, src, dst, via_pois=None, weight="travel_time"):
    orig = ox.distance.nearest_nodes(G, src.centroid_lon, src.centroid_lat)
    dest = ox.distance.nearest_nodes(G, dst.centroid_lon, dst.centroid_lat)

    if via_pois is not None and not via_pois.empty:
        via_nodes = []
        used_rows = []
        for idx, p in via_pois.iterrows():
            try:
                n = ox.distance.nearest_nodes(G, p.geometry.x, p.geometry.y)
                via_nodes.append(n)
                used_rows.append(idx)
            except Exception:
                pass
        via_pois = via_pois.loc[used_rows]
        seq = [orig] + via_nodes + [dest]
        full=[]
        for u,v in zip(seq[:-1], seq[1:]):
            seg = nx.shortest_path(G, u, v, weight=weight)
            full.extend(seg[:-1])
        full.append(seq[-1])
        return full, via_pois

    return nx.shortest_path(G, orig, dest, weight=weight), via_pois

def draw_route_map(G, route, src, dst, sites_gdf=None,
                   via_pois=None, buffer_km=2, iso_time=1800):

    line = route_linestring(G, route)

    sub = nx.ego_graph(G, route[0], radius=iso_time, distance="travel_time")
    iso_pts = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in sub.nodes]
    if len(iso_pts) > 2:
        iso = Polygon(iso_pts).convex_hull
    elif len(iso_pts) == 2:
        iso = LineString(iso_pts).buffer(buffer_km/111)
    else:
        iso = Point(iso_pts[0]).buffer(buffer_km/111)

    m = folium.Map(
        location=[(src.centroid_lat+dst.centroid_lat)/2,
                  (src.centroid_lon+dst.centroid_lon)/2],
        zoom_start=9
    )

    folium.GeoJson(iso, style_function=lambda _: {
        "fillColor":"#3186cc","color":"#3186cc","fillOpacity":0.18
    }).add_to(m)

    _add_polyline(m, line, color="black", weight=4, opacity=0.9)

    for city,col in [(src,"red"),(dst,"blue")]:
        folium.CircleMarker(
            [city.centroid_lat, city.centroid_lon],
            radius=6, color=col, fill=True,
            tooltip=city.region_name
        ).add_to(m)

    if via_pois is not None and not via_pois.empty:
        for _,p in via_pois.iterrows():
            folium.Marker(
                [p.geometry.y,p.geometry.x],
                icon=folium.Icon(color="orange", icon="star"),
                tooltip=f"경유: {p.POI_NM}"
            ).add_to(m)

    return m, line, iso

for src_code, dst_code in interest_pairs:
    src = regions.loc[regions.region_code==src_code].iloc[0]
    dst = regions.loc[regions.region_code==dst_code].iloc[0]

    pad = 0.2
    Gp = load_graph_segment(src, dst, pad_deg=pad)
    if not Gp.edges:
        print(f"Skip {src.region_name}→{dst.region_name}: no edges")
        continue

    Gp = ox.add_edge_speeds(Gp, fallback=30)
    Gp = ox.add_edge_travel_times(Gp)

    via_df = PAIR_TO_DF[(src_code, dst_code)].copy()

    for col in ("lon","lat","POI_NM"):
        if col not in via_df.columns:
            raise ValueError(f"{(src_code, dst_code)}: '{col}' 컬럼이 route CSV에 필요합니다.")
    via_df["lon"] = pd.to_numeric(via_df["lon"], errors="coerce")
    via_df["lat"] = pd.to_numeric(via_df["lat"], errors="coerce")
    via_df = via_df.dropna(subset=["lon","lat"])
    via_df["geometry"] = via_df.apply(lambda r: Point(r["lon"], r["lat"]), axis=1)

    route, used_via = build_route(Gp, src, dst, via_pois=via_df, weight="travel_time")

    buffer_km, iso_time = (5, 3600) if not used_via.empty else (2, 1800)
    m, line, iso = draw_route_map(Gp, route, src, dst, sites_gdf=None,
                                  via_pois=used_via,
                                  buffer_km=buffer_km,
                                  iso_time=iso_time)
    display(m)

    del Gp, route, m, line, iso, via_df, used_via
    gc.collect()

KeyboardInterrupt: 

# 창원 -- 고성

# 목포 -- 진도

In [57]:
import osmnx as ox
import networkx as nx
import folium
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import linemerge
from shapely import wkt
import math, gc
import pandas as pd


PAIR_TO_DF = {
    (36010, 36470): route_4,
}
interest_pairs = list(PAIR_TO_DF.keys())

def haversine(lon1, lat1, lon2, lat2):
    R=6371000
    φ1,φ2 = math.radians(lat1), math.radians(lat2)
    Δφ,Δλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1-a))

def load_graph_segment(src, dst, pad_deg=0.2):
    north = max(src.centroid_lat, dst.centroid_lat) + pad_deg
    south = min(src.centroid_lat, dst.centroid_lat) - pad_deg
    east  = max(src.centroid_lon, dst.centroid_lon) + pad_deg
    west  = min(src.centroid_lon, dst.centroid_lon) - pad_deg

    try:
        G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
        if G.edges: return G
    except ValueError:
        pass

    mid_lat = (src.centroid_lat + dst.centroid_lat)/2
    mid_lon = (src.centroid_lon + dst.centroid_lon)/2
    half = haversine(src.centroid_lon, src.centroid_lat,
                     dst.centroid_lon, dst.centroid_lat)/2
    for extra in (10000, 50000):
        try:
            G = ox.graph_from_point((mid_lat, mid_lon),
                                    dist=half+extra,
                                    network_type="drive",
                                    simplify=True)
            if G.edges: return G
        except ValueError:
            pass
    return nx.MultiDiGraph()

def route_linestring(G, route):
    try:
        geom = ox.utils_graph.route_to_geometry(G, route)
        if geom.geom_type == "LineString":
            return geom
        if geom.geom_type == "MultiLineString":
            return linemerge(geom)
    except Exception:
        pass
    coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in route]
    if len(coords) >= 2:
        return LineString(coords)
    x, y = coords[0]
    eps = 1e-6
    return LineString([(x, y), (x+eps, y+eps)])

def _add_polyline(m, geom, color="black", weight=5, opacity=0.95):
    if geom.geom_type == "LineString":
        folium.PolyLine([[y, x] for x, y in geom.coords],
                        color=color, weight=weight, opacity=opacity).add_to(m)
    elif geom.geom_type == "MultiLineString":
        for g in geom.geoms:
            if g.geom_type == "LineString":
                folium.PolyLine([[y, x] for x, y in g.coords],
                                color=color, weight=weight, opacity=opacity).add_to(m)

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_via_df(df):
    out = df.copy()

    if "geometry" in out.columns and out["geometry"].astype(str).str.contains("^POINT", regex=True).any():
        out["geometry"] = out["geometry"].apply(
            lambda s: wkt.loads(s) if isinstance(s, str) and s.startswith("POINT") else s
        )

    name_col = _pick_col(out, ["POI_NM","POI_NAME","명칭"])
    if name_col is None:
        raise ValueError("경유지 CSV에 POI_NM/POI_NAME/명칭 중 하나가 필요합니다.")
    if name_col != "POI_NM":
        out["POI_NM"] = out[name_col].astype(str)

    lon_col = _pick_col(out, ["lon","LC_LO","x","X","longitude","LONG","경도"])
    lat_col = _pick_col(out, ["lat","LC_LA","y","Y","latitude","LAT","위도"])

    if (lon_col is None or lat_col is None) and "geometry" in out.columns:
        if out["geometry"].apply(lambda g: hasattr(g, "x") and hasattr(g, "y")).any():
            out["lon"] = out["geometry"].apply(lambda g: g.x if hasattr(g, "x") else pd.NA)
            out["lat"] = out["geometry"].apply(lambda g: g.y if hasattr(g, "y") else pd.NA)
            lon_col, lat_col = "lon", "lat"

    if lon_col is None or lat_col is None:
        raise ValueError("경유지 CSV에 경도/위도 컬럼(lon/lat 또는 LC_LO/LC_LA 등)이 필요합니다.")

    out["lon"] = pd.to_numeric(out[lon_col], errors="coerce")
    out["lat"] = pd.to_numeric(out[lat_col], errors="coerce")
    out = out.dropna(subset=["lon","lat"]).copy()
    out["geometry"] = out.apply(lambda r: Point(r["lon"], r["lat"]), axis=1)
    return out

def build_route(G, src, dst, via_pois=None, weight="travel_time"):
    orig = ox.distance.nearest_nodes(G, src.centroid_lon, src.centroid_lat)
    dest = ox.distance.nearest_nodes(G, dst.centroid_lon, dst.centroid_lat)

    if via_pois is not None and not via_pois.empty:
        via_nodes, used_rows = [], []
        for idx, p in via_pois.iterrows():  # CSV 순서 유지
            try:
                n = ox.distance.nearest_nodes(G, p.geometry.x, p.geometry.y)
                via_nodes.append(n); used_rows.append(idx)
            except Exception:
                pass
        via_used = via_pois.loc[used_rows].copy()
        seq = [orig] + via_nodes + [dest]
        full=[]
        for u, v in zip(seq[:-1], seq[1:]):
            seg = nx.shortest_path(G, u, v, weight=weight)
            full.extend(seg[:-1])
        full.append(seq[-1])
        return full, via_used

    return nx.shortest_path(G, orig, dest, weight=weight), pd.DataFrame(columns=["POI_NM","geometry"])

def draw_route_map(G, route, src, dst, sites_gdf=None,
                   via_pois=None, buffer_km=2, iso_time=1800):
    line = route_linestring(G, route)

    sub = nx.ego_graph(G, route[0], radius=iso_time, distance="travel_time")
    iso_pts = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in sub.nodes]
    if len(iso_pts) > 2:
        iso = Polygon(iso_pts).convex_hull
    elif len(iso_pts) == 2:
        iso = LineString(iso_pts).buffer(buffer_km/111)
    else:
        iso = Point(iso_pts[0]).buffer(buffer_km/111)

    m = folium.Map(
        location=[(src.centroid_lat+dst.centroid_lat)/2,
                  (src.centroid_lon+dst.centroid_lon)/2],
        zoom_start=9
    )
    folium.GeoJson(iso, style_function=lambda _: {
        "fillColor":"#3186cc","color":"#3186cc","fillOpacity":0.18
    }).add_to(m)
    _add_polyline(m, line, color="black", weight=5, opacity=0.95)

    for city,col in [(src,"red"),(dst,"blue")]:
        folium.CircleMarker([city.centroid_lat, city.centroid_lon],
                            radius=6, color=col, fill=True,
                            tooltip=city.region_name).add_to(m)

    if via_pois is not None and not via_pois.empty:
        for _, p in via_pois.iterrows():
            folium.Marker([p.geometry.y, p.geometry.x],
                          icon=folium.Icon(color="orange", icon="star"),
                          tooltip=f"경유: {p.POI_NM}").add_to(m)

    return m, line, iso

PAD_BY_PAIR = {
    (22020, 37370): 0.5,
    (33040, 33370): 0.5,
}

for src_code, dst_code in interest_pairs:
    src = regions.loc[regions.region_code==src_code].iloc[0]
    dst = regions.loc[regions.region_code==dst_code].iloc[0]

    pad = PAD_BY_PAIR.get((src_code, dst_code), 0.2)
    Gp = load_graph_segment(src, dst, pad_deg=pad)
    if not Gp.edges:
        print(f"Skip {src.region_name}→{dst.region_name}: no edges")
        continue

    Gp = ox.add_edge_speeds(Gp, fallback=30)
    Gp = ox.add_edge_travel_times(Gp)

    via_raw = PAIR_TO_DF[(src_code, dst_code)]
    via_df = normalize_via_df(via_raw)

    route, used_via = build_route(Gp, src, dst, via_pois=via_df, weight="travel_time")

    expect = list(via_df["POI_NM"].astype(str))
    used = set(used_via["POI_NM"].astype(str)) if not used_via.empty else set()
    dropped = [nm for nm in expect if nm not in used]
    if dropped:
        print(f"[경고] 매칭 실패로 제외된 경유지: {dropped}")

    if route_linestring(Gp, route).length < 1e-5:
        print(f"[주의] {src.region_name}→{dst.region_name} 경로 짧음. pad 확장 재시도")
        pad2 = max(0.5, pad + 0.3)
        Gp2 = load_graph_segment(src, dst, pad_deg=pad2)
        if Gp2.edges:
            Gp2 = ox.add_edge_speeds(Gp2, fallback=30)
            Gp2 = ox.add_edge_travel_times(Gp2)
            route, used_via = build_route(Gp2, src, dst, via_pois=via_df, weight="travel_time")
            Gp = Gp2

    buffer_km, iso_time = (5, 3600) if not used_via.empty else (2, 1800)
    m, line, iso = draw_route_map(Gp, route, src, dst, sites_gdf=None,
                                  via_pois=used_via,
                                  buffer_km=buffer_km, iso_time=iso_time)
    display(m)

    del Gp, route, m, line, iso, via_df, used_via
    gc.collect()

In [54]:
route_1.head(1)

,POI_ID,POI_NM,src_code,dst_code,src_region,dst_region,lon,lat,dist_on_route_m
0,493475,대구역관광안내소,22020,37370,대구광역시 동구,경상북도 고령군,128.595726,35.875751,15221.60151


In [55]:
route_4.head(1)

,POI_ID,POI_NM,MLSFC_NM,CL_NM,CTPRVN_NM,SIGNGU_NM,CL_CD,LC_LO,LC_LA,GID_CD,지역 코드,시도 코드,시군구 코드,교통인프라지수_norm,geometry,dist_on_route,src_region,dst_region,type
0,2957212,여객선매표소,관광지,관광안내소/매표소,전라남도,목포시,60401,126.385176,34.782894,나라980437,36010,36,10,-0.337632,POINT (126.3851759 34.78289439),0.009453,전라남도 목포시,전라남도 진도군,on_route
